# 03 - Understand each family and choose its natural key

This notebook looks at one account at a time. Background: the institution sometimes splits one physical holding across several `investment_id`s in the same sync. A good natural key groups those fragments back into one holding. So groups with more than one id are expected and desired. They are the reason the key exists.

Three tools:

1. `check_family(family, ACCOUNT_ID)`. Counts how often the candidate key groups more than one `investment_id` in the same snapshot.
2. `show_group(family, ACCOUNT_ID, n)`. Shows the full payloads of the fragments in one group, side by side, and marks which fields differ. Use it to judge the key: if only quantities and amounts differ, the group is one holding split in lots and the key is right. If identity fields differ (isin, ticker, issuer, dates), the key merged different instruments and is too coarse.
3. `spec_vs_data(family, ACCOUNT_ID)`. One table with every field of the family, joining the official Open Finance Brasil OpenAPI spec (type, format, required, enum) with what the payloads actually contain for this account (coverage, sample value). Use it to learn the data shape and draft the canonical schema.

How to use:

1. Run the Setup and Spec cells.
2. Set `ACCOUNT_ID`, or run the candidates cell to pick a different account.
3. Run the per-family cells.
4. Change `ACCOUNT_ID` and re-run to compare accounts.

## Setup

In [1]:
import duckdb, json
import pandas as pd
from collections import defaultdict
from pathlib import Path

# Render dataframes in full. No "..." truncation of columns or values.
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 250)

RAW_DATA_DIR = Path('../data/raw')
conn = duckdb.connect()
conn.execute(f"CREATE VIEW raw_positions AS SELECT * FROM read_parquet('{RAW_DATA_DIR / 'raw_positions.parquet'}')")

def sql(query):
    return conn.sql(query).df()


# Candidate natural key per family: the payload fields that should identify
# one physical holding. Edit here to test a different key.
KEYS = {
    'VARIABLE_INCOMES':     ['isinCode', 'ticker'],
    'FUNDS':                ['cnpjNumber'],
    'BANK_FIXED_INCOMES':   ['isinCode', 'issuerInstitutionCnpjNumber', 'issueDate', 'dueDate'],
    'CREDIT_FIXED_INCOMES': ['isinCode', 'debtorCnpjNumber', 'dueDate'],
    'TREASURE_TITLES':      ['isinCode'],
}


def key_groups(family, account_id):
    """Group this account's detail records by the candidate key, snapshot by snapshot.

    Returns one row per (snapshot, key value), with how many distinct
    investment_ids the institution delivered for that key.
    """
    key_fields = ',\n             '.join(
        f"json_extract_string(payload_json, '$.data.{field}') AS \"{field}\""
        for field in KEYS[family]
    )
    query = f"""
      SELECT snapshot_created_at, snapshot_id, account_id,
             {key_fields},
             count(DISTINCT investment_id)     AS n_ids,
             array_agg(DISTINCT investment_id) AS investment_ids
      FROM raw_positions
      WHERE investment_type = '{family}'
        AND payload_kind    = 'detail'
        AND account_id      = '{account_id}'
      GROUP BY ALL
      ORDER BY snapshot_created_at
    """
    return conn.sql(query).df()


def check_family(family, account_id):
    """Summarize how often the candidate key groups more than one investment_id.

    Groups with n_ids > 1 are expected. They are the fragments the key regroups.
    """
    groups = key_groups(family, account_id)
    multi = groups[groups['n_ids'] > 1]
    max_fragments = int(groups['n_ids'].max()) if len(groups) else 0

    print(f"[{family}] account={account_id}")
    print(f"  key fields:                      {', '.join(KEYS[family])}")
    print(f"  holdings (snapshot x key):       {len(groups)}")
    print(f"  with more than 1 investment_id:  {len(multi)}  (expected: fragments the key regroups)")
    print(f"  max fragments in one holding:    {max_fragments}")

    if multi.empty:
        print("  The key never grouped anything. Either this account has no fragmentation,")
        print("  or the key is too fine. Showing plain groups instead.")
        return groups.head(10)
    print("  Sample below: most fragmented holdings first.")
    return multi.sort_values(['n_ids', 'snapshot_created_at'], ascending=[False, True]).head(10)


def flatten_json(obj, prefix=''):
    """Yield (path, value) for every leaf in a nested JSON object.

    Lists are represented by their first element, with [] appended to the path.
    """
    if isinstance(obj, dict):
        for key, value in obj.items():
            yield from flatten_json(value, f'{prefix}.{key}' if prefix else key)
    elif isinstance(obj, list):
        if obj:
            yield from flatten_json(obj[0], prefix + '[]')
        else:
            yield prefix, None
    else:
        yield prefix, obj


def show_group(family, account_id, n=0):
    """Show the fragments of one multi-fragment holding side by side.

    Picks the n-th group, ordered from most to least fragmented. Each column is
    one investment_id (first 8 chars). Each row is one field, from both the
    detail and the balances payloads. The differs column is True where the
    fragments disagree.

    How to judge the key with this view:
      only quantity/amount fields differ -> one holding split in lots, key is right
      identity fields differ             -> key merged different instruments, too coarse
    """
    groups = key_groups(family, account_id)
    multi = groups[groups['n_ids'] > 1].sort_values(['n_ids', 'snapshot_created_at'], ascending=[False, True])
    if multi.empty:
        print(f"[{family}] no multi-fragment holdings for this account.")
        return None

    group = multi.iloc[n]
    ids_sql = ', '.join(f"'{i}'" for i in group['investment_ids'])
    payloads = conn.sql(f"""
        SELECT payload_kind, investment_id, payload_json
        FROM raw_positions
        WHERE snapshot_id   = '{group['snapshot_id']}'
          AND account_id    = '{account_id}'
          AND investment_id IN ({ids_sql})
    """).fetchall()

    fields_by_fragment = defaultdict(dict)
    for kind, investment_id, raw in payloads:
        for path, value in flatten_json(json.loads(raw).get('data', {})):
            fields_by_fragment[investment_id[:8]][f'{kind}.{path}'] = value

    table = pd.DataFrame(fields_by_fragment).sort_index()
    table['differs'] = table.astype(str).nunique(axis=1) > 1

    key_desc = '  '.join(f"{field}={group[field]}" for field in KEYS[family])
    print(f"[{family}] snapshot={group['snapshot_created_at']}  fragments={len(group['investment_ids'])}")
    print(f"  {key_desc}")
    return table


def observed_columns(family, account_id):
    """List every JSON path that appears with a value in this account's payloads.

    Coverage is the share of payloads (per endpoint) where the field is filled.
    """
    payloads = conn.sql(f"""
        SELECT payload_kind, payload_json
        FROM raw_positions
        WHERE investment_type = '{family}' AND account_id = '{account_id}'
    """).fetchall()

    payload_count = defaultdict(int)   # endpoint -> number of payloads
    filled_count = defaultdict(int)    # (endpoint, path) -> payloads where the field has a value
    samples = {}

    for endpoint, raw in payloads:
        payload_count[endpoint] += 1
        seen_here = set()
        for path, value in flatten_json(json.loads(raw).get('data', {})):
            if value is None or value == '':
                continue
            key = (endpoint, path)
            if key in seen_here:
                continue
            seen_here.add(key)
            filled_count[key] += 1
            samples.setdefault(key, value)

    rows = [{
        'endpoint':     endpoint,
        'path':         path,
        'coverage_pct': round(100 * count / payload_count[endpoint], 1),
        'filled':       count,
        'payloads':     payload_count[endpoint],
        'sample':       str(samples[(endpoint, path)])[:80],
    } for (endpoint, path), count in filled_count.items()]
    return pd.DataFrame(rows, columns=['endpoint', 'path', 'coverage_pct', 'filled', 'payloads', 'sample'])


## Spec

Loads the official Open Finance Brasil OpenAPI specs, the source of truth for each family's schema. Files are cached in `notebooks/.ofb_specs/` and downloaded on first use.

In [2]:
import yaml, urllib.request

SPEC_CACHE = Path('.ofb_specs')
SPEC_CACHE.mkdir(exist_ok=True)
SPEC_REPO = 'https://github.com/OpenBanking-Brasil/draft-openapi/blob/main/swagger-apis'
SPEC_RAW  = 'https://raw.githubusercontent.com/OpenBanking-Brasil/draft-openapi/main/swagger-apis'

SPEC_VERSIONS = {
    'VARIABLE_INCOMES':     ('variable-incomes',     '1.3.0'),
    'FUNDS':                ('funds',                '1.1.0'),
    'BANK_FIXED_INCOMES':   ('bank-fixed-incomes',   '1.1.0'),
    'CREDIT_FIXED_INCOMES': ('credit-fixed-incomes', '1.1.0'),
    'TREASURE_TITLES':      ('treasure-titles',      '1.1.0'),
}

# Where each endpoint's data schema lives inside the parsed YAML.
# Some families declare it as a named schema, others inline it under the response.
SPEC_DATA_SCHEMAS = {
    ('VARIABLE_INCOMES',     'balances'): ('components', 'schemas', 'ResponseVariableIncomesBalanceData'),
    ('VARIABLE_INCOMES',     'detail'):   ('components', 'schemas', 'ResponseVariableIncomesProductIdentificationData'),
    ('FUNDS',                'balances'): ('components', 'schemas', 'ResponseFundsBalanceData'),
    ('FUNDS',                'detail'):   ('components', 'schemas', 'ResponseFundsProductIdentificationData'),
    ('BANK_FIXED_INCOMES',   'balances'): ('components', 'schemas', 'ResponseBankFixedIncomesBalances', 'properties', 'data'),
    ('BANK_FIXED_INCOMES',   'detail'):   ('components', 'schemas', 'IdentifyProduct'),
    ('CREDIT_FIXED_INCOMES', 'balances'): ('components', 'schemas', 'ResponseCreditFixedIncomesBalances', 'properties', 'data'),
    ('CREDIT_FIXED_INCOMES', 'detail'):   ('components', 'schemas', 'CreditFixedIdentification'),
    ('TREASURE_TITLES',      'balances'): ('components', 'schemas', 'TreasureTitlesBalances'),
    ('TREASURE_TITLES',      'detail'):   ('components', 'schemas', 'TreasureTitlesIdentifyProduct'),
}

_loaded_specs = {}

def load_spec(family):
    """Parse the family's OpenAPI YAML, downloading it on first use."""
    if family not in _loaded_specs:
        slug, version = SPEC_VERSIONS[family]
        path = SPEC_CACHE / f'{slug}-{version}.yml'
        if not path.exists():
            path.write_bytes(urllib.request.urlopen(f'{SPEC_RAW}/{slug}/{version}.yml').read())
        _loaded_specs[family] = yaml.safe_load(path.read_text())
    return _loaded_specs[family]


def resolve_ref(family, node):
    """Follow $ref links until we reach a real schema node."""
    schemas = load_spec(family)['components']['schemas']
    while isinstance(node, dict) and '$ref' in node:
        node = schemas[node['$ref'].split('/')[-1]]
    return node


def spec_fields(family, node, prefix='', required=False):
    """Yield one dict per leaf field declared in a spec schema node."""
    node = resolve_ref(family, node)
    if 'allOf' in node:
        for part in node['allOf']:
            yield from spec_fields(family, part, prefix, required)
    elif node.get('type') == 'array':
        yield from spec_fields(family, node.get('items', {}), prefix + '[]', required)
    elif 'properties' in node:
        required_names = set(node.get('required', []))
        for name, child in node['properties'].items():
            path = f'{prefix}.{name}' if prefix else name
            yield from spec_fields(family, child, path, name in required_names)
    else:
        yield {
            'path':      prefix,
            'spec_type': node.get('type'),
            'format':    node.get('format'),
            'pattern':   node.get('pattern'),
            'enum':      ', '.join(map(str, node.get('enum', []))) or None,
            'required':  required,
        }


def spec_schema(family, endpoint):
    """The spec's declared schema for one endpoint, one row per field."""
    node = load_spec(family)
    for step in SPEC_DATA_SCHEMAS[(family, endpoint)]:
        node = node[step]
    return pd.DataFrame(spec_fields(family, node))


def spec_vs_data(family, account_id):
    """Every field of the family: the spec's declaration joined with observed data.

    The status column tells you where to look:
      NEVER_SEEN   declared in the spec, never filled in this account's payloads
      NOT_IN_SPEC  present in the data but not declared in the spec
      ok           declared and filled
    """
    spec_parts = []
    for endpoint in ('balances', 'detail'):
        part = spec_schema(family, endpoint)
        part.insert(0, 'endpoint', endpoint)
        spec_parts.append(part)
    spec = pd.concat(spec_parts, ignore_index=True)

    observed = observed_columns(family, account_id)
    table = spec.merge(observed, on=['endpoint', 'path'], how='outer')

    def status(row):
        if pd.isna(row['spec_type']):
            return 'NOT_IN_SPEC'
        if pd.isna(row['coverage_pct']):
            return 'NEVER_SEEN'
        return 'ok'
    table['status'] = table.apply(status, axis=1)

    slug, version = SPEC_VERSIONS[family]
    print(f"[{family}] account={account_id}")
    print(f"  spec: {SPEC_REPO}/{slug}/{version}.yml")
    return table.sort_values(['endpoint', 'status', 'path']).reset_index(drop=True)


for family in SPEC_VERSIONS:
    load_spec(family)
print('specs loaded for:', ', '.join(SPEC_VERSIONS))


specs loaded for: VARIABLE_INCOMES, FUNDS, BANK_FIXED_INCOMES, CREDIT_FIXED_INCOMES, TREASURE_TITLES


## Pick account

In [3]:
ACCOUNT_ID = 'e4c413e3-83d9-5973-b390-5b7a9fb93140'


In [4]:
# Helper: list top N accounts by row volume, so you can grab another one.
sql("""
SELECT account_id,
       count(DISTINCT investment_type) AS families,
       count(*)                        AS detail_rows
FROM raw_positions
WHERE payload_kind = 'detail'
GROUP BY 1
ORDER BY families DESC, detail_rows DESC
LIMIT 20
""")


,account_id,families,detail_rows
0,e4c413e3-83d9-5973-b390-5b7a9fb93140,5,2326
1,f3ce7b7a-d660-52cf-b819-c10cb66c1338,5,2159
2,6b8fcd56-8159-51fd-820d-8aa84292931c,5,1740
3,951f1b05-df5e-59dd-a07d-33b771ad6e91,5,1680
4,2ab892a8-1694-5b17-a13f-eb758d48210c,5,1664
5,36c13618-4ea4-5208-bf90-8b9ce29d3b32,5,1611
6,ccdc8073-a0e3-5799-992d-9af6310ba197,5,1427
7,70f3d5e8-c50c-5c44-845b-7269d93b05e6,5,1423
8,b08a2b3e-6031-5955-9c37-78b6bd96ca12,5,1118
9,e645159d-7697-50df-a024-93fc2b6a991a,5,1071


## Key check per family

For each family: how many holdings exist for this account, and how many of them the institution delivered as more than one `investment_id`. Re-run after changing `ACCOUNT_ID`.

In [5]:
  sql("""
      SELECT * FROM raw_positions
      WHERE investment_id IN (
          '1f4422cf-c626-5414-bc3e-279e88c0c30d',
          '643ebd0b-af21-59df-82bc-7b0c6212fd3a'
      )
  """).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67
institution_id,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003,00000000000003
institution_name,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau,Itau
party_id,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61,21c6ad56-3ebc-5881-9b5e-507fa27cea61

In [6]:
check_family('VARIABLE_INCOMES', ACCOUNT_ID)

[VARIABLE_INCOMES] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  key fields:                      isinCode, ticker
  holdings (snapshot x key):       170
  with more than 1 investment_id:  136  (expected: fragments the key regroups)
  max fragments in one holding:    2
  Sample below: most fragmented holdings first.


,snapshot_created_at,snapshot_id,account_id,isinCode,ticker,n_ids,investment_ids
1,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRB3SAACNOR6,B3SA3,2,"[7155a1bd-4b85-54ba-9172-abeef31b40cd, a2cb2a52-228b-500b-a29e-3b9d257673a0]"
2,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBPACUNT002,BPAC11,2,"[b1d32cfe-e1c6-519a-acf8-04587ba4f9ba, 8d006f6e-b4f0-5df5-9945-e7a3b7424394]"
3,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRWEGEACNOR0,WEGE3,2,"[e1003357-c686-5601-9514-32b9ee109ce7, 0936fcb2-49b4-5764-af8c-d08be5ecbbd9]"
4,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBBDCACNPR8,BBDC4,2,"[bf7af325-1a83-5224-bb28-3e8ad25ae879, 06714758-cc12-58d0-a302-60a07daa8a3e]"
5,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRHASHCTF001,HASH11,2,"[5a4b1f1e-a854-5e55-8b4a-92d36708ff32, 50704ad1-08e0-5a2c-acd6-0eb1e66a8fc5]"
7,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRPETRACNPR6,PETR4,2,"[1f4422cf-c626-5414-bc3e-279e88c0c30d, 643ebd0b-af21-59df-82bc-7b0c6212fd3a]"
8,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRVALEACNOR0,VALE3,2,"[27154b2a-3ca7-5d21-aa5b-00098f3c1950, 04770487-117f-5a4e-b3d2-2c6cd893ead4]"
9,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRITUBACNPR1,ITUB4,2,"[9939e377-1eaa-5213-916f-d57835c08767, cd4c1ff2-d9ea-560e-8098-73048848555c]"
10,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRVALEACNOR0,VALE3,2,"[27154b2a-3ca7-5d21-aa5b-00098f3c1950, 04770487-117f-5a4e-b3d2-2c6cd893ead4]"
11,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRB3SAACNOR6,B3SA3,2,"[a2cb2a52-228b-500b-a29e-3b9d257673a0, 7155a1bd-4b85-54ba-9172-abeef31b40cd]"


In [7]:
check_family('FUNDS', ACCOUNT_ID)

[FUNDS] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  key fields:                      cnpjNumber
  holdings (snapshot x key):       108
  with more than 1 investment_id:  54  (expected: fragments the key regroups)
  max fragments in one holding:    2
  Sample below: most fragmented holdings first.


,snapshot_created_at,snapshot_id,account_id,cnpjNumber,n_ids,investment_ids
2,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,32915987000180,2,"[bb197213-3043-51d3-861c-d6293a1e2ecb, b4bcc670-ee4f-5ddb-a8f2-2575f860a7c3]"
4,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,18318213000154,2,"[10664e75-2472-5418-8884-e8b680009c4b, d8f753f4-c1d1-59ec-8747-72ed24a50fff]"
5,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,11052478000102,2,"[d016f3a5-ea0b-5493-88db-5915670a25ef, 35ee83ae-006a-51c8-9bd5-c1e4fe803928]"
8,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,32915987000180,2,"[bb197213-3043-51d3-861c-d6293a1e2ecb, b4bcc670-ee4f-5ddb-a8f2-2575f860a7c3]"
10,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,18318213000154,2,"[d8f753f4-c1d1-59ec-8747-72ed24a50fff, 10664e75-2472-5418-8884-e8b680009c4b]"
11,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,11052478000102,2,"[d016f3a5-ea0b-5493-88db-5915670a25ef, 35ee83ae-006a-51c8-9bd5-c1e4fe803928]"
15,2026-08-01T09:15:58Z,4ef5ccf7-70e7-5b9a-9b70-a0e2e79c2df2,e4c413e3-83d9-5973-b390-5b7a9fb93140,18318213000154,2,"[d8f753f4-c1d1-59ec-8747-72ed24a50fff, 10664e75-2472-5418-8884-e8b680009c4b]"
16,2026-08-01T09:15:58Z,4ef5ccf7-70e7-5b9a-9b70-a0e2e79c2df2,e4c413e3-83d9-5973-b390-5b7a9fb93140,32915987000180,2,"[b4bcc670-ee4f-5ddb-a8f2-2575f860a7c3, bb197213-3043-51d3-861c-d6293a1e2ecb]"
17,2026-08-01T09:15:58Z,4ef5ccf7-70e7-5b9a-9b70-a0e2e79c2df2,e4c413e3-83d9-5973-b390-5b7a9fb93140,11052478000102,2,"[35ee83ae-006a-51c8-9bd5-c1e4fe803928, d016f3a5-ea0b-5493-88db-5915670a25ef]"
18,2026-08-02T09:15:59Z,d4193448-4868-54f3-b423-3b11194e042d,e4c413e3-83d9-5973-b390-5b7a9fb93140,18318213000154,2,"[d8f753f4-c1d1-59ec-8747-72ed24a50fff, 10664e75-2472-5418-8884-e8b680009c4b]"


In [8]:
check_family('BANK_FIXED_INCOMES', ACCOUNT_ID)

[BANK_FIXED_INCOMES] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  key fields:                      isinCode, issuerInstitutionCnpjNumber, issueDate, dueDate
  holdings (snapshot x key):       1513
  with more than 1 investment_id:  17  (expected: fragments the key regroups)
  max fragments in one holding:    2
  Sample below: most fragmented holdings first.


,snapshot_created_at,snapshot_id,account_id,isinCode,issuerInstitutionCnpjNumber,issueDate,dueDate,n_ids,investment_ids
62,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[9443318b-867c-5125-ab20-9d37c18629ef, df43c178-6919-530c-ad26-83688c640689]"
105,2026-08-01T09:15:58Z,4ef5ccf7-70e7-5b9a-9b70-a0e2e79c2df2,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[df43c178-6919-530c-ad26-83688c640689, 9443318b-867c-5125-ab20-9d37c18629ef]"
256,2026-08-02T09:15:59Z,d4193448-4868-54f3-b423-3b11194e042d,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[9443318b-867c-5125-ab20-9d37c18629ef, df43c178-6919-530c-ad26-83688c640689]"
274,2026-08-03T09:16:00Z,4542eb44-e32a-56f1-acef-5ed7ba160e96,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[9443318b-867c-5125-ab20-9d37c18629ef, df43c178-6919-530c-ad26-83688c640689]"
393,2026-08-04T09:16:01Z,4c5d13d8-20d5-55b2-80af-ae198771c98c,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[9443318b-867c-5125-ab20-9d37c18629ef, df43c178-6919-530c-ad26-83688c640689]"
493,2026-08-05T09:16:02Z,a14afd1a-eb73-56f5-a19c-ad15a8a6db19,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[9443318b-867c-5125-ab20-9d37c18629ef, df43c178-6919-530c-ad26-83688c640689]"
534,2026-08-08T09:16:03Z,92eae213-f465-5e4f-b284-2cb08945ae2d,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[9443318b-867c-5125-ab20-9d37c18629ef, df43c178-6919-530c-ad26-83688c640689]"
725,2026-08-09T09:16:04Z,e2e5dd1f-4110-5a76-a17a-7e27ef9a3dd8,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[9443318b-867c-5125-ab20-9d37c18629ef, df43c178-6919-530c-ad26-83688c640689]"
747,2026-08-09T09:16:04Z,143a6c16-d34a-54b1-8e51-ab7df66bcdd8,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[df43c178-6919-530c-ad26-83688c640689, 9443318b-867c-5125-ab20-9d37c18629ef]"
842,2026-08-10T09:16:05Z,df0c71a8-b941-547d-a777-9eb98288ad11,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRBANKCDB4A4,00360305000104,2025-12-05,2028-12-01,2,"[df43c178-6919-530c-ad26-83688c640689, 9443318b-867c-5125-ab20-9d37c18629ef]"


In [9]:
check_family('CREDIT_FIXED_INCOMES', ACCOUNT_ID)

[CREDIT_FIXED_INCOMES] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  key fields:                      isinCode, debtorCnpjNumber, dueDate
  holdings (snapshot x key):       72
  with more than 1 investment_id:  18  (expected: fragments the key regroups)
  max fragments in one holding:    2
  Sample below: most fragmented holdings first.


,snapshot_created_at,snapshot_id,account_id,isinCode,debtorCnpjNumber,dueDate,n_ids,investment_ids
2,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[773f768a-f537-50f9-b48d-b902ca2e8eb0, 099dabbb-9c7a-5af1-b534-016baeaaf2ab]"
7,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[773f768a-f537-50f9-b48d-b902ca2e8eb0, 099dabbb-9c7a-5af1-b534-016baeaaf2ab]"
11,2026-08-01T09:15:58Z,4ef5ccf7-70e7-5b9a-9b70-a0e2e79c2df2,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[099dabbb-9c7a-5af1-b534-016baeaaf2ab, 773f768a-f537-50f9-b48d-b902ca2e8eb0]"
13,2026-08-02T09:15:59Z,d4193448-4868-54f3-b423-3b11194e042d,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[773f768a-f537-50f9-b48d-b902ca2e8eb0, 099dabbb-9c7a-5af1-b534-016baeaaf2ab]"
17,2026-08-03T09:16:00Z,4542eb44-e32a-56f1-acef-5ed7ba160e96,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[099dabbb-9c7a-5af1-b534-016baeaaf2ab, 773f768a-f537-50f9-b48d-b902ca2e8eb0]"
23,2026-08-04T09:16:01Z,4c5d13d8-20d5-55b2-80af-ae198771c98c,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[099dabbb-9c7a-5af1-b534-016baeaaf2ab, 773f768a-f537-50f9-b48d-b902ca2e8eb0]"
27,2026-08-05T09:16:02Z,a14afd1a-eb73-56f5-a19c-ad15a8a6db19,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[099dabbb-9c7a-5af1-b534-016baeaaf2ab, 773f768a-f537-50f9-b48d-b902ca2e8eb0]"
31,2026-08-08T09:16:03Z,92eae213-f465-5e4f-b284-2cb08945ae2d,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[099dabbb-9c7a-5af1-b534-016baeaaf2ab, 773f768a-f537-50f9-b48d-b902ca2e8eb0]"
34,2026-08-09T09:16:04Z,143a6c16-d34a-54b1-8e51-ab7df66bcdd8,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[099dabbb-9c7a-5af1-b534-016baeaaf2ab, 773f768a-f537-50f9-b48d-b902ca2e8eb0]"
35,2026-08-09T09:16:04Z,e2e5dd1f-4110-5a76-a17a-7e27ef9a3dd8,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRDEBSEC01A1,09149503000106,2031-06-15,2,"[773f768a-f537-50f9-b48d-b902ca2e8eb0, 099dabbb-9c7a-5af1-b534-016baeaaf2ab]"


In [10]:
check_family('TREASURE_TITLES', ACCOUNT_ID)

[TREASURE_TITLES] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  key fields:                      isinCode
  holdings (snapshot x key):       136
  with more than 1 investment_id:  102  (expected: fragments the key regroups)
  max fragments in one holding:    2
  Sample below: most fragmented holdings first.


,snapshot_created_at,snapshot_id,account_id,isinCode,n_ids,investment_ids
0,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRSTNCNTB6A3,2,"[696ea744-627c-50bc-9558-55ec1bc07fe7, e07aee78-2499-52ef-a294-622e9add454f]"
1,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRSTNCLF1RU6,2,"[664ea616-95eb-50ae-8009-031728994189, 66a0f5cf-e288-5819-9f9e-3dbd2c1b0c04]"
2,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRSTNCLTN8D1,2,"[bf4240fa-e0fc-5532-98fe-69f457c05e1b, 1cba4c59-c9f4-542d-b03b-0601f17d2e10]"
4,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,,2,"[6a2683c1-de20-59cf-b8e1-302b04e4f7ee, 8565739b-2399-5ed4-8a1e-25c726a3d9bc]"
5,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRSTNCNTB2U0,2,"[26df1e0f-ec84-5419-9855-5c548665d629, 2ba20725-85d8-5c58-bc81-ab482390e6ef]"
7,2026-07-30T09:15:56Z,f0d5ee73-d13f-51b9-b6a7-59b74b0800cd,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRSTNCNAP0G8,2,"[5b1473fb-ef67-53f0-8b27-876a6a75bea2, 84c76927-2ef1-56f7-aad9-a4f18e6ce778]"
8,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRSTNCNTB6A3,2,"[696ea744-627c-50bc-9558-55ec1bc07fe7, e07aee78-2499-52ef-a294-622e9add454f]"
9,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRSTNCLF1RU6,2,"[66a0f5cf-e288-5819-9f9e-3dbd2c1b0c04, 664ea616-95eb-50ae-8009-031728994189]"
10,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,BRSTNCLTN8D1,2,"[1cba4c59-c9f4-542d-b03b-0601f17d2e10, bf4240fa-e0fc-5532-98fe-69f457c05e1b]"
12,2026-07-31T09:15:57Z,b2b77380-5e0a-5c56-88a7-d491a4a050ba,e4c413e3-83d9-5973-b390-5b7a9fb93140,,2,"[6a2683c1-de20-59cf-b8e1-302b04e4f7ee, 8565739b-2399-5ed4-8a1e-25c726a3d9bc]"


## Drill into one group: partition or collision?

`show_group` prints the fragments of one holding side by side, one column per `investment_id`, one row per field. The `differs` column is True where the fragments disagree.

How to read it: if only quantities and amounts differ, the fragments are one holding split in lots. The key is right and canonical should sum them. If identity fields differ, the key is too coarse.

Change the family string or `n` (0 = most fragmented group) to inspect others.

In [11]:
show_group('VARIABLE_INCOMES', ACCOUNT_ID, n=0)

[VARIABLE_INCOMES] snapshot=2026-07-30T09:15:56Z  fragments=2
  isinCode=BRHASHCTF001  ticker=HASH11


,50704ad1,5a4b1f1e,differs
balances.blockedBalance.amount,0.00,0.00,False
balances.blockedBalance.currency,BRL,BRL,False
balances.closingPrice.amount,64.16,64.99,True
balances.closingPrice.currency,BRL,BRL,False
balances.grossAmount.amount,60505.24,101587.18,True
balances.grossAmount.currency,BRL,BRL,False
balances.priceFactor,1.00,1.00,False
balances.quantity,943.00000000,1563.00000000,True
balances.referenceDate,2026-07-30,2026-07-30,False
detail.isinCode,BRHASHCTF001,BRHASHCTF001,False


## Spec vs data: the full schema of each family

One table per family. Every field from the official OpenAPI spec (type, format, required, enum) joined with what this account's payloads actually contain (coverage, sample value).

Read the `status` column first:

- `NEVER_SEEN`: the spec declares the field but this account never fills it
- `NOT_IN_SPEC`: the payload carries a field the spec does not declare
- `ok`: declared and filled

High-coverage fields are the candidates for first-class canonical columns.

In [12]:
spec_vs_data('VARIABLE_INCOMES', ACCOUNT_ID)

[VARIABLE_INCOMES] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  spec: https://github.com/OpenBanking-Brasil/draft-openapi/blob/main/swagger-apis/variable-incomes/1.3.0.yml


,endpoint,path,spec_type,format,pattern,enum,required,coverage_pct,filled,payloads,sample,status
0,balances,blockedBalance.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,306,306,0.00,ok
1,balances,blockedBalance.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,306,306,BRL,ok
2,balances,closingPrice.amount,string,double,"^-?\d{1,15}\.\d{2,4}$",None,True,100.0,306,306,53.06,ok
3,balances,closingPrice.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,306,306,BRL,ok
4,balances,grossAmount.amount,string,double,"^-?\d{1,15}\.\d{2,4}$",None,True,100.0,306,306,236612.38,ok
5,balances,grossAmount.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,306,306,BRL,ok
6,balances,priceFactor,string,double,"^\d{1,15}\.\d{2,8}$",None,True,100.0,306,306,1.00,ok
7,balances,quantity,string,double,"^-?\d{1,15}\.\d{2,8}$",None,True,100.0,306,306,4459.00000000,ok
8,balances,referenceDate,string,date,^(\d{4})-(1[0-2]|0?[1-9])-(3[01]|[12][0-9]|0?[1-9])$,None,True,100.0,306,306,2026-07-30,ok
9,detail,isinCode,string,None,^[A-Z]{2}([A-Z0-9]){9}\d{1}$,None,True,88.9,272,306,BRB3SAACNOR6,ok


In [13]:
spec_vs_data('FUNDS', ACCOUNT_ID)

[FUNDS] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  spec: https://github.com/OpenBanking-Brasil/draft-openapi/blob/main/swagger-apis/funds/1.1.0.yml


,endpoint,path,spec_type,format,pattern,enum,required,coverage_pct,filled,payloads,sample,status
0,balances,blockedAmount.amount,string,double,"^-?\d{1,15}\.\d{2,4}$",None,True,88.9,144.0,162.0,0.00,ok
1,balances,blockedAmount.currency,string,NaN,^[A-Z]{3}$,None,True,88.9,144.0,162.0,BRL,ok
2,balances,financialTransactionTaxProvision.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,88.9,144.0,162.0,0.00,ok
3,balances,financialTransactionTaxProvision.currency,string,NaN,^[A-Z]{3}$,None,True,88.9,144.0,162.0,BRL,ok
4,balances,grossAmount.amount,string,double,"^-?\d{1,15}\.\d{2,4}$",None,True,88.9,144.0,162.0,77021.38,ok
5,balances,grossAmount.currency,string,NaN,^[A-Z]{3}$,None,True,88.9,144.0,162.0,BRL,ok
6,balances,incomeTaxProvision.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,88.9,144.0,162.0,192.55,ok
7,balances,incomeTaxProvision.currency,string,NaN,^[A-Z]{3}$,None,True,88.9,144.0,162.0,BRL,ok
8,balances,netAmount.amount,string,double,"^-?\d{1,15}\.\d{2,4}$",None,True,88.9,144.0,162.0,76828.83,ok
9,balances,netAmount.currency,string,NaN,^[A-Z]{3}$,None,True,88.9,144.0,162.0,BRL,ok


In [14]:
spec_vs_data('BANK_FIXED_INCOMES', ACCOUNT_ID)

[BANK_FIXED_INCOMES] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  spec: https://github.com/OpenBanking-Brasil/draft-openapi/blob/main/swagger-apis/bank-fixed-incomes/1.1.0.yml


,endpoint,path,spec_type,format,pattern,enum,required,coverage_pct,filled,payloads,sample,status
0,balances,blockedBalance.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,1564.0,1564.0,0.00,ok
1,balances,blockedBalance.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,1564.0,1564.0,BRL,ok
2,balances,financialTransactionTax.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,1564.0,1564.0,0.00,ok
3,balances,financialTransactionTax.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,1564.0,1564.0,BRL,ok
4,balances,grossAmount.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,1564.0,1564.0,42822.72,ok
5,balances,grossAmount.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,1564.0,1564.0,BRL,ok
6,balances,incomeTax.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,1564.0,1564.0,107.06,ok
7,balances,incomeTax.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,1564.0,1564.0,BRL,ok
8,balances,netAmount.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,1564.0,1564.0,42715.66,ok
9,balances,netAmount.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,1564.0,1564.0,BRL,ok


In [15]:
spec_vs_data('CREDIT_FIXED_INCOMES', ACCOUNT_ID)

[CREDIT_FIXED_INCOMES] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  spec: https://github.com/OpenBanking-Brasil/draft-openapi/blob/main/swagger-apis/credit-fixed-incomes/1.1.0.yml


,endpoint,path,spec_type,format,pattern,enum,required,coverage_pct,filled,payloads,sample,status
0,balances,fine.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,NaN,NaN,NaN,NaN,NEVER_SEEN
1,balances,fine.currency,string,NaN,^[A-Z]{3}$,None,True,NaN,NaN,NaN,NaN,NEVER_SEEN
2,balances,latePayment.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,NaN,NaN,NaN,NaN,NEVER_SEEN
3,balances,latePayment.currency,string,NaN,^[A-Z]{3}$,None,True,NaN,NaN,NaN,NaN,NEVER_SEEN
4,balances,preFixedRate,string,NaN,^-?\d{1}\.\d{6}$,None,False,NaN,NaN,NaN,NaN,NEVER_SEEN
5,balances,blockedBalance.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,90.0,90.0,0.00,ok
6,balances,blockedBalance.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,90.0,90.0,BRL,ok
7,balances,financialTransactionTax.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,90.0,90.0,0.00,ok
8,balances,financialTransactionTax.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,90.0,90.0,BRL,ok
9,balances,grossAmount.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,90.0,90.0,157644.69,ok


In [16]:
spec_vs_data('TREASURE_TITLES', ACCOUNT_ID)

[TREASURE_TITLES] account=e4c413e3-83d9-5973-b390-5b7a9fb93140
  spec: https://github.com/OpenBanking-Brasil/draft-openapi/blob/main/swagger-apis/treasure-titles/1.1.0.yml


,endpoint,path,spec_type,format,pattern,enum,required,coverage_pct,filled,payloads,sample,status
0,balances,blockedBalance.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,238.0,238.0,0.00,ok
1,balances,blockedBalance.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,238.0,238.0,BRL,ok
2,balances,financialTransactionTax.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,238.0,238.0,0.00,ok
3,balances,financialTransactionTax.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,238.0,238.0,BRL,ok
4,balances,grossAmount.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,238.0,238.0,51255.28,ok
5,balances,grossAmount.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,238.0,238.0,BRL,ok
6,balances,incomeTax.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,238.0,238.0,128.14,ok
7,balances,incomeTax.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,238.0,238.0,BRL,ok
8,balances,netAmount.amount,string,double,"^\d{1,15}\.\d{2,4}$",None,True,100.0,238.0,238.0,51127.14,ok
9,balances,netAmount.currency,string,NaN,^[A-Z]{3}$,None,True,100.0,238.0,238.0,BRL,ok


## Modeling draft: raw → canonical → consumption

Three layers, built live below:

- **raw** — `raw_positions` stays as ingested: one row per S3 object, every column VARCHAR, payload untouched. Its job is audit and replay; a typed raw layer breaks the day the institution ships a malformed value.
- **canonical** — one table per family (`canon_*`), one row per **lot** (`snapshot_id`, `investment_id`): the lot's `detail` and `balances` payloads joined and flattened into typed columns (DECIMAL for money and quantities, DATE/TIMESTAMP for time). All parsing and casting happens here, once.
- **consumption** — views only. `holdings_*` regroups lots into physical holdings by the natural keys this notebook validated (additive fields sum across lots). `portfolio` unions the five families for cross-family questions.

Canonical stays at lot grain instead of holding grain because fragments carry real information — the WEGE3 group above has two lots with different closing prices. Regrouping is cheap in a view; un-summing is impossible.

In [17]:
# Canonical layer: one table per family, one row per lot (snapshot_id, investment_id).
# Each entry: column -> (payload kind, path under $.data, SQL type).
MONEY, QTY, RATE = 'DECIMAL(18,4)', 'DECIMAL(24,8)', 'DECIMAL(12,6)'

FIXED_INCOME_BALANCES = {
    'quantity':            ('balances', 'quantity',                       QTY),
    'updated_unit_price':  ('balances', 'updatedUnitPrice.amount',        MONEY),
    'purchase_unit_price': ('balances', 'purchaseUnitPrice.amount',       MONEY),
    'gross_amount':        ('balances', 'grossAmount.amount',             MONEY),
    'net_amount':          ('balances', 'netAmount.amount',               MONEY),
    'blocked_amount':      ('balances', 'blockedBalance.amount',          MONEY),
    'income_tax':          ('balances', 'incomeTax.amount',               MONEY),
    'transaction_tax':     ('balances', 'financialTransactionTax.amount', MONEY),
    'currency':            ('balances', 'grossAmount.currency',           'VARCHAR'),
    'reference_datetime':  ('balances', 'referenceDateTime',              'TIMESTAMP'),
}
REMUNERATION = {
    'indexer':                ('detail', 'remuneration.indexer',                    'VARCHAR'),
    'pre_fixed_rate':         ('detail', 'remuneration.preFixedRate',               RATE),
    'post_fixed_indexer_pct': ('detail', 'remuneration.postFixedIndexerPercentage', RATE),
}

CANONICAL_FIELDS = {
    'VARIABLE_INCOMES': {
        'isin_code':      ('detail',   'isinCode',                    'VARCHAR'),
        'ticker':         ('detail',   'ticker',                      'VARCHAR'),
        'issuer_cnpj':    ('detail',   'issuerInstitutionCnpjNumber', 'VARCHAR'),
        'quantity':       ('balances', 'quantity',                    QTY),
        'closing_price':  ('balances', 'closingPrice.amount',         MONEY),
        'price_factor':   ('balances', 'priceFactor',                 RATE),
        'gross_amount':   ('balances', 'grossAmount.amount',          MONEY),
        'blocked_amount': ('balances', 'blockedBalance.amount',       MONEY),
        'currency':       ('balances', 'grossAmount.currency',        'VARCHAR'),
        'reference_date': ('balances', 'referenceDate',               'DATE'),
    },
    'FUNDS': {
        'cnpj_number':     ('detail',   'cnpjNumber',                  'VARCHAR'),
        'fund_name':       ('detail',   'name',                        'VARCHAR'),
        'anbima_category': ('detail',   'anbimaCategory',              'VARCHAR'),
        'quota_quantity':  ('balances', 'quotaQuantity',               QTY),
        'quota_price':     ('balances', 'quotaGrossPriceValue.amount', MONEY),
        'gross_amount':    ('balances', 'grossAmount.amount',          MONEY),
        'net_amount':      ('balances', 'netAmount.amount',            MONEY),
        'blocked_amount':  ('balances', 'blockedAmount.amount',        MONEY),
        'income_tax':      ('balances', 'incomeTaxProvision.amount',   MONEY),
        'transaction_tax': ('balances', 'financialTransactionTaxProvision.amount', MONEY),
        'currency':        ('balances', 'grossAmount.currency',        'VARCHAR'),
        'reference_date':  ('balances', 'referenceDate',               'DATE'),
    },
    'BANK_FIXED_INCOMES': {
        'product_type':      ('detail', 'investmentType',              'VARCHAR'),
        'isin_code':         ('detail', 'isinCode',                    'VARCHAR'),
        'issuer_cnpj':       ('detail', 'issuerInstitutionCnpjNumber', 'VARCHAR'),
        'clearing_code':     ('detail', 'clearingCode',                'VARCHAR'),
        'issue_date':        ('detail', 'issueDate',                   'DATE'),
        'due_date':          ('detail', 'dueDate',                     'DATE'),
        'grace_period_date': ('detail', 'gracePeriodDate',             'DATE'),
        'purchase_date':     ('detail', 'purchaseDate',                'DATE'),
        'issue_unit_price':  ('detail', 'issueUnitPrice.amount',       MONEY),
        **REMUNERATION,
        **FIXED_INCOME_BALANCES,
    },
    'CREDIT_FIXED_INCOMES': {
        'product_type':     ('detail', 'investmentType',              'VARCHAR'),
        'isin_code':        ('detail', 'isinCode',                    'VARCHAR'),
        'debtor_cnpj':      ('detail', 'debtorCnpjNumber',            'VARCHAR'),
        'debtor_name':      ('detail', 'debtorName',                  'VARCHAR'),
        'issuer_cnpj':      ('detail', 'issuerInstitutionCnpjNumber', 'VARCHAR'),
        'clearing_code':    ('detail', 'clearingCode',                'VARCHAR'),
        'issue_date':       ('detail', 'issueDate',                   'DATE'),
        'due_date':         ('detail', 'dueDate',                     'DATE'),
        'purchase_date':    ('detail', 'purchaseDate',                'DATE'),
        'issue_unit_price': ('detail', 'issueUnitPrice.amount',       MONEY),
        'tax_exempt':       ('detail', 'taxExemptProduct',            'VARCHAR'),
        **REMUNERATION,
        **FIXED_INCOME_BALANCES,
    },
    'TREASURE_TITLES': {
        'isin_code':     ('detail', 'isinCode',     'VARCHAR'),
        'product_name':  ('detail', 'productName',  'VARCHAR'),
        'due_date':      ('detail', 'dueDate',      'DATE'),
        'purchase_date': ('detail', 'purchaseDate', 'DATE'),
        **REMUNERATION,
        **FIXED_INCOME_BALANCES,
    },
}

ENVELOPE = ['institution_id', 'institution_name', 'party_id', 'account_id', 'connection_id']

def build_canonical(family):
    """CREATE TABLE canon_<family>: detail FULL JOIN balances per lot, typed columns.

    FULL JOIN because some lots arrive with balances only (no detail payload).
    Asserts the result has exactly one row per lot seen in raw.
    """
    table = f'canon_{family.lower()}'
    envelope = ',\n               '.join(f'coalesce(d.{c}, b.{c}) AS {c}' for c in ENVELOPE)
    typed = ',\n               '.join(
        f"TRY_CAST(json_extract_string({payload[0]}.payload_json, '$.data.{path}') AS {sql_type}) AS {col}"
        for col, (payload, path, sql_type) in CANONICAL_FIELDS[family].items()
    )
    conn.execute(f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT snapshot_id, investment_id,
               TRY_CAST(coalesce(d.snapshot_created_at, b.snapshot_created_at) AS TIMESTAMP) AS snapshot_created_at,
               {envelope},
               {typed}
        FROM (SELECT * FROM raw_positions WHERE investment_type = '{family}' AND payload_kind = 'detail') AS d
        FULL JOIN (SELECT * FROM raw_positions WHERE investment_type = '{family}' AND payload_kind = 'balances') AS b
        USING (snapshot_id, investment_id)
    """)
    rows, dup = conn.sql(f'SELECT count(*), count(*) - count(DISTINCT (snapshot_id, investment_id)) FROM {table}').fetchone()
    raw_lots = conn.sql(f"SELECT count(DISTINCT (snapshot_id, investment_id)) FROM raw_positions WHERE investment_type = '{family}'").fetchone()[0]
    assert dup == 0 and rows == raw_lots, (family, rows, raw_lots, dup)
    return rows

for family in CANONICAL_FIELDS:
    print(f'canon_{family.lower():<22} {build_canonical(family):>7} lots')

canon_variable_incomes         17570 lots
canon_funds                     8355 lots
canon_bank_fixed_incomes       66183 lots
canon_credit_fixed_incomes      5447 lots
canon_treasure_titles           9138 lots


### Consumption

Views only, no tables. Each `holdings_*` view regroups lots into physical holdings using the natural keys validated above: additive fields sum, identity fields are constant within a group (that is what this notebook verified). `portfolio` puts the five families behind one schema for cross-family questions. Materialize only if the views ever get slow.

In [18]:
# family -> (natural key columns, instrument label, additive columns).
# Same keys as KEYS at the top, in canonical column names.
HOLDINGS = {
    'VARIABLE_INCOMES':     (['isin_code', 'ticker'],                                'ticker',       ['quantity', 'gross_amount', 'blocked_amount']),
    'FUNDS':                (['cnpj_number'],                                        'fund_name',    ['quota_quantity', 'gross_amount', 'net_amount', 'blocked_amount']),
    'BANK_FIXED_INCOMES':   (['isin_code', 'issuer_cnpj', 'issue_date', 'due_date'], 'isin_code',    ['quantity', 'gross_amount', 'net_amount', 'blocked_amount']),
    'CREDIT_FIXED_INCOMES': (['isin_code', 'debtor_cnpj', 'due_date'],               'isin_code',    ['quantity', 'gross_amount', 'net_amount', 'blocked_amount']),
    'TREASURE_TITLES':      (['isin_code'],                                          'product_name', ['quantity', 'gross_amount', 'net_amount', 'blocked_amount']),
}

# ponytail: lots missing the detail payload have NULL key columns and lump into
# one group per snapshot; give them a fallback key (investment_id) if that matters.
union_parts = []
for family, (keys, label, sums) in HOLDINGS.items():
    name = family.lower()
    label_col = '' if label in keys else f'any_value({label}) AS {label},'
    conn.execute(f"""
        CREATE OR REPLACE VIEW holdings_{name} AS
        SELECT snapshot_created_at, snapshot_id, account_id, institution_name,
               {', '.join(keys)},
               {label_col}
               {', '.join(f'sum({c}) AS {c}' for c in sums)},
               any_value(currency)      AS currency,
               count(*)                 AS n_lots,
               array_agg(investment_id) AS investment_ids
        FROM canon_{name}
        GROUP BY ALL
    """)
    union_parts.append(f"""
        SELECT snapshot_created_at, snapshot_id, account_id, institution_name,
               '{family}' AS investment_type, {label} AS instrument,
               {sums[0]} AS quantity, gross_amount, currency, n_lots
        FROM holdings_{name}""")

conn.execute('CREATE OR REPLACE VIEW portfolio AS' + ' UNION ALL'.join(union_parts))
sql("SELECT investment_type, count(*) AS holding_rows, sum(n_lots) AS lots FROM portfolio GROUP BY 1 ORDER BY 1")

,investment_type,holding_rows,lots
0,BANK_FIXED_INCOMES,64448,66183.0
1,CREDIT_FIXED_INCOMES,5280,5447.0
2,FUNDS,7705,8355.0
3,TREASURE_TITLES,8401,9138.0
4,VARIABLE_INCOMES,15150,17570.0


In [19]:
# The WEGE3 case from show_group above: two lots with different prices -> one holding.
sql(f"""
    SELECT snapshot_created_at, ticker, quantity, gross_amount, n_lots, investment_ids
    FROM holdings_variable_incomes
    WHERE account_id = '{ACCOUNT_ID}' AND ticker = 'WEGE3'
    ORDER BY snapshot_created_at
    LIMIT 5
""")

,snapshot_created_at,ticker,quantity,gross_amount,n_lots,investment_ids
0,2026-07-30 09:15:56,WEGE3,169.0,6589.71,2,"[e1003357-c686-5601-9514-32b9ee109ce7, 0936fcb2-49b4-5764-af8c-d08be5ecbbd9]"
1,2026-07-31 09:15:57,WEGE3,169.0,6632.06,2,"[e1003357-c686-5601-9514-32b9ee109ce7, 0936fcb2-49b4-5764-af8c-d08be5ecbbd9]"
2,2026-08-01 09:15:58,WEGE3,169.0,6838.05,2,"[e1003357-c686-5601-9514-32b9ee109ce7, 0936fcb2-49b4-5764-af8c-d08be5ecbbd9]"
3,2026-08-03 09:16:00,WEGE3,169.0,6716.91,2,"[e1003357-c686-5601-9514-32b9ee109ce7, 0936fcb2-49b4-5764-af8c-d08be5ecbbd9]"
4,2026-08-04 09:16:01,WEGE3,169.0,6793.40,2,"[e1003357-c686-5601-9514-32b9ee109ce7, 0936fcb2-49b4-5764-af8c-d08be5ecbbd9]"


In [20]:
# Cross-family: the account's latest portfolio in one query.
sql(f"""
    SELECT investment_type, count(*) AS holdings, sum(n_lots) AS lots, sum(gross_amount) AS gross_amount
    FROM portfolio
    WHERE account_id = '{ACCOUNT_ID}'
      AND snapshot_created_at = (SELECT max(snapshot_created_at) FROM portfolio WHERE account_id = '{ACCOUNT_ID}')
    GROUP BY 1
    ORDER BY gross_amount DESC
""")

,investment_type,holdings,lots,gross_amount
0,BANK_FIXED_INCOMES,90,92.0,10483749.94
1,VARIABLE_INCOMES,10,18.0,2108724.26
2,TREASURE_TITLES,8,14.0,1282126.26
3,FUNDS,6,9.0,1006617.25
4,CREDIT_FIXED_INCOMES,4,5.0,923223.92
